In [3]:
pip install Xgboost antropy nbformat>=4.2.0 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Import the required libraries 

In [4]:
#Import all the required libraries for this notebook
import os, glob, re, copy, warnings, sys
import numpy as np
import pandas as pd
from math import gcd
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")


from scipy.signal import (butter, sosfiltfilt, iirnotch,
                          welch, resample_poly, tf2sos)
from scipy.stats  import kurtosis, skew

from sklearn.pipeline            import Pipeline
from sklearn.preprocessing       import StandardScaler
from sklearn.feature_selection   import SelectKBest, f_classif
from sklearn.model_selection     import (LeaveOneGroupOut, GroupKFold,
                                          RandomizedSearchCV)
from sklearn.linear_model        import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm                 import SVC
from sklearn.ensemble            import RandomForestClassifier
from sklearn.metrics             import (accuracy_score, balanced_accuracy_score,
                                          roc_auc_score, confusion_matrix,
                                          f1_score)


Set the Input directory, default values

In [ ]:
#Set the Read and write path
HC_PATH  = r"D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\Healthy subjects\Healthy Subjects - EEG_csv_format"
AD_PATH  = r"D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\AD subjects\AD subjects - EEG_csv_format"
OUT_DIR  = r"D:\ASDS\Capstone Project\Combined Data"

#Here are some of the constant values which will be used for Data Preprocessing
FS_RAW     = 512      # original sampling rate (Hz)
FS_TARGET  = 256      # downsample target (Hz)
EPOCH_SEC  = 4        # epoch window length (seconds)
OVERLAP    = 0.5      # epoch overlap fraction (0.5 = 50%)
HIGHPASS   = 0.5      # bandpass low cut (Hz)
LOWPASS    = 45.0     # bandpass high cut (Hz)
NOTCH_HZ   = 50.0     # notch frequency — change to 60.0 for US recordings
AMP_THRESH = 200.0    # artifact rejection threshold (µV)
LPC_ORDER  = 7        # LPC model order (per LEAPD paper)
N_FEAT_SEL = 80       # top-K features selected inside each LOSO fold

#Classify the signal into this bands for better understanding
BANDS = {
    "delta": (0.5,  4.0),
    "theta": (4.0,  8.0),
    "alpha": (8.0, 13.0),
    "beta" : (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

# 18 shared channels (T3→T7, T4→T8 aliased automatically)
COMMON_18 = ["F7","Fp1","Fp2","F8","F3","Fz","F4","C3","Cz","P8","P7","P4","T7","P3","O1","O2","C4","T8"]

# Legacy alias mapping
ALIAS = {"T3": "T7", "T4": "T8"}

# LEAPD best-performing electrodes (from Nature PD 2024 paper)
LEAPD_CH = ["P8","P7","P4","F4","C3","Cz","C4","O1"]


Signal Preprocessing

In [ ]:
#These functions are the preprocessing techniques for our EEG signals

def bandpass(data, lo, hi, fs, order=4):
    sos = butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, data, axis=0)


def notch(data, freq, fs, q=30):
    b, a = iirnotch(freq / (fs / 2.0), q)
    return sosfiltfilt(tf2sos(b, a), data, axis=0)


def downsample(data, fs_in, fs_out):
    if fs_in == fs_out:
        return data
    g = gcd(int(fs_in), int(fs_out))
    return resample_poly(data, int(fs_out)//g, int(fs_in)//g,axis=0).astype(np.float32)


def make_epochs(data, fs, sec, overlap):
    length = int(fs * sec)
    step   = int(length * (1 - overlap))
    return [data[s:s+length] for s in range(0, len(data) - length + 1, step)]


Feature Extraction


In [7]:
# Optional — install with: pip install xgboost antropy shap
try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False
    print("[WARNING] xgboost not installed. Run: pip install xgboost")

try:
    # Prevent occasional numba JIT crashes on some Windows setups.
    os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
    from antropy import sample_entropy
    ANTROPY_OK = True
except ImportError:
    ANTROPY_OK = False
    print("[WARNING] antropy not installed. Run: pip install antropy")

# antropy can be unstable on some Windows builds; keep it opt-in.
USE_ANTROPY = ANTROPY_OK and os.environ.get("EEG_USE_ANTROPY", "0") == "1"
if ANTROPY_OK and not USE_ANTROPY:
    print("[INFO] antropy detected but disabled by default; set EEG_USE_ANTROPY=1 to enable.")


def _sample_entropy_safe(signal):
    """Use antropy when available; otherwise fall back to a stable proxy."""
    if not USE_ANTROPY:
        return float(kurtosis(signal))
    try:
        return float(sample_entropy(signal, order=2))
    except Exception:
        return float(kurtosis(signal))

#3 Feature Engineering

# ── 3.1  Band Power (log) ─────────────────────────────────────────────────────
def feat_band_power(epoch, fs):
    """18 channels × 5 bands = 90 features"""
    out, nperseg = [], min(epoch.shape[0], fs * 2)
    for ch in range(epoch.shape[1]):
        f, psd = welch(epoch[:, ch], fs=fs, nperseg=nperseg)
        for lo, hi in BANDS.values():
            m = (f >= lo) & (f <= hi)
            out.append(float(np.log1p(np.trapz(psd[m], f[m]))))
    return np.array(out, dtype=np.float32)                      # (90,)


# ── 3.2  Alpha / Theta Ratio ─────────────────────────────────────────────────
def feat_alpha_theta(epoch, fs):
    """18 features — strongest single AD biomarker"""
    out, nperseg = [], min(epoch.shape[0], fs * 2)
    for ch in range(epoch.shape[1]):
        f, psd = welch(epoch[:, ch], fs=fs, nperseg=nperseg)
        a = np.trapezoid(psd[(f>=8)&(f<13)],  f[(f>=8)&(f<13)])  + 1e-10
        t = np.trapezoid(psd[(f>=4)&(f<8)],   f[(f>=4)&(f<8)])   + 1e-10
        out.append(float(np.log(a / t)))
    return np.array(out, dtype=np.float32)                      # (18,)


# ── 3.3  Spectral Entropy ────────────────────────────────────────────────────
def feat_spectral_entropy(epoch, fs):
    """18 features — decreases in AD (brain becomes more ordered)"""
    out, nperseg = [], min(epoch.shape[0], fs * 2)
    for ch in range(epoch.shape[1]):
        f, psd = welch(epoch[:, ch], fs=fs, nperseg=nperseg)
        pn  = psd / (psd.sum() + 1e-10)
        out.append(float(-np.sum(pn * np.log2(pn + 1e-10))))
    return np.array(out, dtype=np.float32)                      # (18,)


# ── 3.4  LPC Coefficients (LEAPD-style) ──────────────────────────────────────
def _levinson(r, order):
    """Levinson-Durbin recursion — returns LPC coefficients of given order."""
    if r[0] == 0:
        return np.zeros(order, np.float32)
    a, e = np.zeros(order), r[0]
    for i in range(order):
        k        = (r[i+1] - np.dot(a[:i], r[1:i+1][::-1])) / (e + 1e-10)
        a_new    = a.copy()
        a_new[i] = k
        for j in range(i):
            a_new[j] = a[j] - k * a[i-1-j]
        a  = a_new
        e *= 1 - k**2
    return a.astype(np.float32)


def lpc_for_channel(sig, fs, order=LPC_ORDER):
    """
    Band-filter signal into 5 BANDS, compute LPC(order) per band.
    Returns: order * 5 = 35 coefficients per channel.
    """
    coeffs = []
    for lo, hi in BANDS.values():
        try:
            sos = butter(4, [lo, hi], btype="band", fs=fs, output="sos")
            s   = sosfiltfilt(sos, sig)
        except Exception:
            s   = sig.copy()
        n = len(s)
        r = np.correlate(s, s, mode="full")[n-1:n+order+1]
        coeffs.extend(_levinson(r, order))
    return np.array(coeffs, np.float32)                         # (35,)


def feat_lpc_all(epoch, fs):
    """All 18 channels × 35 = 630 LPC features"""
    return np.concatenate([lpc_for_channel(epoch[:, ch], fs)
                           for ch in range(epoch.shape[1])])    # (630,)


def feat_lpc_leapd(epoch, fs, ch_names=COMMON_18):
    """LEAPD-style: 8 best posterior electrodes × 35 = 280 LPC features"""
    indices = [ch_names.index(c) for c in LEAPD_CH if c in ch_names]
    return np.concatenate([lpc_for_channel(epoch[:, i], fs)
                           for i in indices])                   # (280,)


# ── 3.5  Functional Connectivity ─────────────────────────────────────────────
def feat_connectivity(epoch):
    """Upper triangle of 18×18 Pearson correlation matrix = 153 features"""
    C   = np.corrcoef(epoch.T)
    idx = np.triu_indices(epoch.shape[1], k=1)
    return C[idx].astype(np.float32)                            # (153,)


# ── 3.6  Nonlinear / Complexity Features ─────────────────────────────────────
def feat_nonlinear(epoch):
    """
    Per channel (18):
      - Sample Entropy  (or kurtosis fallback if antropy missing)
      - Hjorth Mobility
      - Hjorth Complexity
      - Skewness
    Total: 18 × 4 = 72 features
    """
    out = []
    for ch in range(epoch.shape[1]):
        s     = epoch[:, ch].astype(np.float64)
        d1    = np.diff(s);  d2 = np.diff(d1)
        v0    = np.var(s)  + 1e-10
        v1    = np.var(d1) + 1e-10
        v2    = np.var(d2) + 1e-10
        mob   = float(np.sqrt(v1 / v0))
        comp  = float(np.sqrt(v2 / v1) / mob) if mob > 0 else 0.0
        se    = _sample_entropy_safe(s)
        sk    = float(skew(s))
        out.extend([se, mob, comp, sk])
    return np.array(out, dtype=np.float32)                      # (72,)


# ── 3.7  Master Extractor ─────────────────────────────────────────────────────
def extract_features(epoch, fs):
    """
    Concatenate all feature groups for one epoch.
    Band Power       :  90
    Alpha/Theta      :  18
    Spectral Entropy :  18
    LPC all ch       : 630
    LPC LEAPD-8ch    : 280
    Connectivity     : 153
    Nonlinear        :  72
    ──────────────────────
    TOTAL            : 1261
    """
    parts = [
        feat_band_power(epoch, fs),
        feat_alpha_theta(epoch, fs),
        feat_spectral_entropy(epoch, fs),
        feat_lpc_all(epoch, fs),
        feat_lpc_leapd(epoch, fs),
        feat_connectivity(epoch),
        feat_nonlinear(epoch),
    ]
    vec = np.concatenate(parts)
    # Replace NaN/Inf with 0
    vec = np.where(np.isfinite(vec), vec, 0.0)
    return vec              

[INFO] antropy detected but disabled by default; set EEG_USE_ANTROPY=1 to enable.


4.Load Data

In [8]:
# get the channel mapping
def _get_ch_mapping(channels_csv):
    df  = pd.read_csv(channels_csv)
    return {f"Ch{i+1}": n for i, n in enumerate(df["channel_name"].tolist())}

# load the data from the input directory for Data Preprocessing
def load_dataset(hc_path=HC_PATH, ad_path=AD_PATH):
    """
    Scans HC and AD folders, loads every *_trial.csv, preprocesses,
    segments into epochs, extracts features.

    Returns
    -------
    X      : np.ndarray  (total_epochs, 1261)
    y      : np.ndarray  (total_epochs,)   0=HC, 1=AD
    groups : np.ndarray  (total_epochs,)   subject ID strings
    """
    X_list, y_list, g_list = [], [], []

    sources = [(hc_path, 0, "PNT"), (ad_path, 1, "sub")]

    for folder, label, prefix in sources:
        files = sorted(glob.glob(os.path.join(folder, "*_trial.csv")))
        tag   = "HC" if label == 0 else "AD"
        print(f"\n  [{tag}] {len(files)} trial files found in:\n  {folder}")

        for fpath in files:
            fname = os.path.basename(fpath)
            m     = re.match(rf"({prefix})(\d+)_trial", fname, re.IGNORECASE)
            if not m:
                print(f"    [skip] unrecognised filename: {fname}")
                continue

            sid      = f"{tag}_{m.group(1)}{m.group(2)}"
            ch_csv   = fpath.replace("_trial.csv", "_channels.csv")
            print(f"    [run] {sid} ...")

            if not os.path.exists(ch_csv):
                print(f"    [skip] {sid} — _channels.csv missing")
                continue

            # ── Load and rename columns ────────────────────────────────────
            df = pd.read_csv(fpath)
            df.rename(columns=_get_ch_mapping(ch_csv), inplace=True)
            df.rename(columns=ALIAS, inplace=True)

            missing = [c for c in COMMON_18 if c not in df.columns]
            if missing:
                print(f"    [skip] {sid} — missing channels: {missing}")
                continue

            raw = df[COMMON_18].values.astype(np.float32)

            # ── Preprocessing ──────────────────────────────────────────────
            sig = downsample(raw, FS_RAW, FS_TARGET)
            sig = bandpass(sig, HIGHPASS, LOWPASS, FS_TARGET)
            sig = notch(sig, NOTCH_HZ, FS_TARGET)

            # ── Epoch → feature extraction ─────────────────────────────────
            epochs   = make_epochs(sig, FS_TARGET, EPOCH_SEC, OVERLAP)
            n_valid  = 0
            for ep in epochs:
                ep64 = np.array(ep, dtype=np.float64)
                if np.any(np.abs(ep64) > AMP_THRESH):   # artifact rejection
                    continue
                feat = extract_features(ep64, FS_TARGET)
                X_list.append(feat)
                y_list.append(label)
                g_list.append(sid)
                n_valid += 1

            print(f"    ✓ {sid} | {n_valid}/{len(epochs)} valid epochs")

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list,  dtype=np.int32)
    g = np.array(g_list)

    print(f"\n{'='*55}")
    print(f"  DATASET LOADED")
    print(f"  X shape         : {X.shape}")
    print(f"  Features/epoch  : {X.shape[1]}")
    print(f"  AD epochs  (y=1): {(y==1).sum()}")
    print(f"  HC epochs  (y=0): {(y==0).sum()}")
    print(f"  Subjects        : {len(set(g))}")
    print(f"{'='*55}")
    return X, y, g


In [10]:
def _run_simple(train_idx, test_idx, X, y, pipeline):
    """One LOSO fold — no inner CV (used for LDA)."""
    p = copy.deepcopy(pipeline)
    p.fit(X[train_idx], y[train_idx])
    prob = float(p.predict_proba(X[test_idx])[:, 1].mean())
    return int(y[test_idx][0]), int(prob >= 0.4), prob


def _run_tuned(train_idx, test_idx, X, y, groups, base_pipe,
               param_dist, n_iter=20):
    """One LOSO fold — with inner GroupKFold hyperparameter tuning."""
    Xtr, Xte = X[train_idx], X[test_idx]
    ytr       = y[train_idx]
    gtr       = groups[train_idx]

    search = RandomizedSearchCV(
        copy.deepcopy(base_pipe),
        param_distributions = param_dist,
        n_iter              = n_iter,
        cv                  = GroupKFold(n_splits=5).split(Xtr, ytr, gtr),
        scoring             = "balanced_accuracy",
        n_jobs              = 1,
        random_state        = 42,
        refit               = True,
        error_score         = 0.0
    )
    search.fit(Xtr, ytr)
    prob = float(search.best_estimator_.predict_proba(Xte)[:, 1].mean())
    return int(y[test_idx][0]), int(prob >= 0.5), prob


def _evaluate(name, results):
    """Aggregate per-subject results into metrics."""
    yt  = [r[0] for r in results]
    yp  = [r[1] for r in results]
    ypr = [r[2] for r in results]

    cm         = confusion_matrix(yt, yp)
    tn, fp, fn, tp_val = cm.ravel()

    acc  = accuracy_score(yt, yp)
    bacc = balanced_accuracy_score(yt, yp)
    try:
        auc = roc_auc_score(yt, ypr)
    except Exception:
        auc = 0.0
    f1   = f1_score(yt, yp, average="binary")
    sens = tp_val  / (tp_val + fn + 1e-10)
    spec = tn      / (tn + fp + 1e-10)

    print(f"\n{'='*55}")
    print(f"  MODEL : {name}")
    print(f"{'='*55}")
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Balanced Accuracy : {bacc:.4f}")
    print(f"  ROC-AUC           : {auc:.4f}")
    print(f"  F1 Score          : {f1:.4f}")
    print(f"  Sensitivity       : {sens:.4f}  (true AD detection rate)")
    print(f"  Specificity       : {spec:.4f}  (true HC detection rate)")
    print(f"  Confusion Matrix  :\n       Pred HC  Pred AD")
    print(f"  True HC  {cm[0,0]:>6}   {cm[0,1]:>6}")
    print(f"  True AD  {cm[1,0]:>6}   {cm[1,1]:>6}")

    return {"model": name, "accuracy": round(acc,4), "balanced_acc": round(bacc,4),
            "auc": round(auc,4), "f1": round(f1,4),
            "sensitivity": round(sens,4), "specificity": round(spec,4),
            "cm": cm, "y_true": yt, "y_pred": yp, "y_prob": ypr}


Model Building

In [11]:
#Perform the scaling, Selecting the k bestfeatures and build the machine learning model
_steps = [("scaler", StandardScaler()),
          ("select", SelectKBest(f_classif, k=N_FEAT_SEL))]

MODELS = {
    # Logistic Regression 
    "Logistic Regression": {
        "pipe" : Pipeline(_steps + [("clf", LogisticRegression(max_iter=2000,
                                                                random_state=42))]),
        "params": {"select__k": [60, 80, 100],
                   "clf__C"   : [0.01, 0.1, 1, 10, 100]},
        "n_iter": 10,
    },
    #  LDA 
    "LDA": {
        "pipe" : Pipeline(_steps + [("clf", LinearDiscriminantAnalysis(
                                              solver="lsqr", shrinkage="auto"))]),
        "params": None,
        "n_iter": None,
    },
    #  SVM RBF 
    "SVM RBF": {
        "pipe" : Pipeline(_steps + [("clf", SVC(kernel="rbf", probability=True,
                                                 random_state=42, cache_size=500))]),
        "params": {"select__k" : [60, 80, 100],
                   "clf__C"    : [0.01, 0.1, 1, 10, 100],
                   "clf__gamma": ["scale", "auto", 0.001, 0.01]},
        "n_iter": 20,
    },
    #  Random Forest 
    "Random Forest": {
        "pipe" : Pipeline(_steps + [("clf", RandomForestClassifier(random_state=42, n_jobs=1))]),
        "params": {"select__k"             : [60, 80, 100],
                   "clf__n_estimators"     : [100, 200, 300],
                   "clf__max_depth"        : [None, 5, 10, 20],
                   "clf__min_samples_split": [2, 5, 10]},
        "n_iter": 20,
    },
}

if XGBOOST_OK:
    MODELS["XGBoost"] = {
        "pipe" : Pipeline(_steps + [("clf", XGBClassifier(
                                              eval_metric="logloss",
                                              random_state=42, n_jobs=1))]),
        "params": {"select__k"          : [60, 80, 100],
                   "clf__n_estimators"  : [100, 200, 300],
                   "clf__max_depth"     : [3, 5, 7],
                   "clf__learning_rate" : [0.01, 0.05, 0.1],
                   "clf__subsample"     : [0.7, 0.8, 1.0]},
        "n_iter": 20,
    }

In [12]:
#LOSO Validation
def run_loso(X, y, groups):
    """
    Run all models under LOSO cross-validation.
    - Simple models (LDA): single pipeline, no inner CV
    - Tuned models: RandomizedSearchCV with inner GroupKFold(5)
    - Parallelised with joblib across subjects
    """
    logo   = LeaveOneGroupOut()
    folds  = list(logo.split(X, y, groups))
    subjs  = sorted(set(groups))
    n_ad   = sum(1 for s in subjs if s.startswith("AD"))
    n_hc   = sum(1 for s in subjs if s.startswith("HC"))

    print(f"\n{'='*55}")
    print(f"  LOSO EVALUATION")
    print(f"  Total folds     : {len(folds)}")
    print(f"  AD subjects     : {n_ad}")
    print(f"  HC subjects     : {n_hc}")
    print(f"  Feature dim     : {X.shape[1]}")
    print(f"  Top-K selected  : {N_FEAT_SEL}")
    print(f"{'='*55}")

    all_results = []

    for mname, cfg in MODELS.items():
        print(f"\n  ▶ Running {mname} ...")

        if cfg["params"] is None:
            raw = Parallel(n_jobs=-1, backend="loky", verbose=2)(
                delayed(_run_simple)(ti, te, X, y, cfg["pipe"])
                for ti, te in folds
            )
        else:
            raw = Parallel(n_jobs=-1, backend="loky", verbose=2)(
                delayed(_run_tuned)(ti, te, X, y, groups,
                                    cfg["pipe"], cfg["params"], cfg["n_iter"])
                for ti, te in folds
            )

        res = _evaluate(mname, raw)
        all_results.append(res)

    return all_results


In [13]:

def print_leaderboard(results):

    '''This function takes the results dictionary and print it in a table format'''
    
    hdr = f"{'Model':<22}{'Acc':>7}{'BalAcc':>9}{'AUC':>8}{'F1':>8}{'Sens':>8}{'Spec':>8}"
    print("\n\n" + "="*65)
    print(f"  FINAL LEADERBOARD — Subject-Level LOSO")
    print("="*65)
    print(hdr)
    print("-"*65)
    for r in sorted(results, key=lambda x: x["auc"], reverse=True):
        print(f"{r['model']:<22}{r['accuracy']:>7.3f}{r['balanced_acc']:>9.3f}"
              f"{r['auc']:>8.3f}{r['f1']:>8.3f}"
              f"{r['sensitivity']:>8.3f}{r['specificity']:>8.3f}")


def save_results(results, out_dir=OUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    rows = [{k: v for k, v in r.items()
             if k not in ("cm","y_true","y_pred","y_prob")}
            for r in results]
    path = os.path.join(out_dir, "loso_results.csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"\n✓ Results saved → {path}")


def plot_confusion_matrices(results, out_dir=OUT_DIR):
    """Save confusion matrix PNG for each model."""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        os.makedirs(out_dir, exist_ok=True)
        n = len(results)
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        if n == 1:
            axes = [axes]
        for ax, r in zip(axes, results):
            sns.heatmap(r["cm"], annot=True, fmt="d", cmap="Blues", ax=ax,
                        xticklabels=["HC","AD"], yticklabels=["HC","AD"])
            ax.set_title(f"{r['model']}\nACC={r['accuracy']:.2f}  "
                         f"AUC={r['auc']:.2f}", fontsize=9)
            ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        plt.tight_layout()
        fig_path = os.path.join(out_dir, "confusion_matrices.png")
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"✓ Confusion matrices → {fig_path}")
    except ImportError:
        print("[INFO] matplotlib/seaborn not installed — skipping plots")


def plot_roc_curves(results, out_dir=OUT_DIR):
    """Save ROC curve PNG for all models."""
    try:
        import matplotlib.pyplot as plt
        from sklearn.metrics import roc_curve
        os.makedirs(out_dir, exist_ok=True)
        plt.figure(figsize=(6, 5))
        for r in results:
            fpr, tpr, _ = roc_curve(r["y_true"], r["y_prob"])
            plt.plot(fpr, tpr, lw=2,
                     label=f"{r['model']} (AUC={r['auc']:.3f})")
        plt.plot([0,1],[0,1],"k--", lw=1)
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curves — Subject-Level LOSO")
        plt.legend(loc="lower right", fontsize=8)
        plt.tight_layout()
        fig_path = os.path.join(out_dir, "roc_curves.png")
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"✓ ROC curves        → {fig_path}")
    except ImportError:
        print("[INFO] matplotlib not installed — skipping ROC plot")


In [ ]:
if __name__ == "__main__":
    '''
    This is the main function,calls the Load,preprocess, Feature Engineer,model build sequentially.    
    '''

    print("=" * 65)
    print("  EEG AD vs HC — LOSO Classification Pipeline")
    print("=" * 65)

    # ── STEP 1: Load data and extract features ──────────────────────────────
    #X, y, groups = load_dataset()

    # ── STEP 2: Save features (skip re-extraction next time) ────────────────
    #os.makedirs(OUT_DIR, exist_ok=True)
    #np.save(os.path.join(OUT_DIR, "X.npy"),      X)
    #np.save(os.path.join(OUT_DIR, "y.npy"),      y)
    #np.save(os.path.join(OUT_DIR, "groups.npy"), groups)
    #print("\n✓ Features saved.")

    # ── To RELOAD saved features instead of re-extracting, uncomment below ──
    OUT_DIR = "D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\only37sampleResults"
    X      = np.load(os.path.join(OUT_DIR, "X_all.npy"))
    y      = np.load(os.path.join(OUT_DIR, "y_all.npy"))
    groups = np.load(os.path.join(OUT_DIR, "g_all.npy"))

    # ── To COMBINE with Dataset 1, uncomment below ──────────────────────────
    # X, y, groups = combine_datasets(X1, y1, g1, X, y, groups)

    # ── STEP 3: Run LOSO for all models ─────────────────────────────────────
    results = run_loso(X, y, groups)

    # ── STEP 4: Print leaderboard ────────────────────────────────────────────
    print_leaderboard(results)

    # ── STEP 5: Save CSV + plots ─────────────────────────────────────────────
    save_results(results)
    plot_confusion_matrices(results)
    plot_roc_curves(results)

    print("\n" + "="*65)
    print("  ✓  PIPELINE COMPLETE")
    print("="*65)   

  EEG AD vs HC — LOSO Classification Pipeline

  LOSO EVALUATION
  Total folds     : 32
  AD subjects     : 15
  HC subjects     : 17
  Feature dim     : 1261
  Top-K selected  : 80

  ▶ Running Logistic Regression ...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 out of  32 | elapsed:   54.2s remaining:   42.2s
[Parallel(n_jobs=-1)]: Done  32 out of  32 | elapsed:   57.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.



  MODEL : Logistic Regression
  Accuracy          : 0.7188
  Balanced Accuracy : 0.7157
  ROC-AUC           : 0.7020
  F1 Score          : 0.6897
  Sensitivity       : 0.6667  (true AD detection rate)
  Specificity       : 0.7647  (true HC detection rate)
  Confusion Matrix  :
       Pred HC  Pred AD
  True HC      13        4
  True AD       5       10

  ▶ Running LDA ...


[Parallel(n_jobs=-1)]: Done  18 out of  32 | elapsed:    0.9s remaining:    0.7s
[Parallel(n_jobs=-1)]: Done  32 out of  32 | elapsed:    1.0s finished



  MODEL : LDA
  Accuracy          : 0.6875
  Balanced Accuracy : 0.6863
  ROC-AUC           : 0.6706
  F1 Score          : 0.6667
  Sensitivity       : 0.6667  (true AD detection rate)
  Specificity       : 0.7059  (true HC detection rate)
  Confusion Matrix  :
       Pred HC  Pred AD
  True HC      12        5
  True AD       5       10

  ▶ Running SVM RBF ...


...


  MODEL : SVM RBF
  Accuracy          : 0.7812
  Balanced Accuracy : 0.7784
  ROC-AUC           : 0.8039
  F1 Score          : 0.7586
  Sensitivity       : 0.7333  (true AD detection rate)
  Specificity       : 0.8235  (true HC detection rate)
  Confusion Matrix  :
       Pred HC  Pred AD
  True HC      14        3
  True AD       4       11

  ▶ Running Random Forest ...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 out of  32 | elapsed: 149.4min remaining: 116.2min
[Parallel(n_jobs=-1)]: Done  32 out of  32 | elapsed: 150.8min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.



  MODEL : Random Forest
  Accuracy          : 0.6562
  Balanced Accuracy : 0.6490
  ROC-AUC           : 0.8078
  F1 Score          : 0.5926
  Sensitivity       : 0.5333  (true AD detection rate)
  Specificity       : 0.7647  (true HC detection rate)
  Confusion Matrix  :
       Pred HC  Pred AD
  True HC      13        4
  True AD       7        8

  ▶ Running XGBoost ...


[Parallel(n_jobs=-1)]: Done  18 out of  32 | elapsed: 26.6min remaining: 20.7min
[Parallel(n_jobs=-1)]: Done  32 out of  32 | elapsed: 27.1min finished



  MODEL : XGBoost
  Accuracy          : 0.6562
  Balanced Accuracy : 0.6490
  ROC-AUC           : 0.7569
  F1 Score          : 0.5926
  Sensitivity       : 0.5333  (true AD detection rate)
  Specificity       : 0.7647  (true HC detection rate)
  Confusion Matrix  :
       Pred HC  Pred AD
  True HC      13        4
  True AD       7        8


  FINAL LEADERBOARD — Subject-Level LOSO
Model                     Acc   BalAcc     AUC      F1    Sens    Spec
-----------------------------------------------------------------
Random Forest           0.656    0.649   0.808   0.593   0.533   0.765
SVM RBF                 0.781    0.778   0.804   0.759   0.733   0.824
XGBoost                 0.656    0.649   0.757   0.593   0.533   0.765
Logistic Regression     0.719    0.716   0.702   0.690   0.667   0.765
LDA                     0.688    0.686   0.671   0.667   0.667   0.706

✓ Results saved → D:\ASDS\Capstone Project\Combined Data\loso_results.csv
[INFO] matplotlib/seaborn not installed — ski

In [1]:
pip install plotly


     ---------------------------------------- 9.9/9.9 MB 15.1 MB/s eta 0:00:00
     ------------------------------------- 449.4/449.4 kB 14.2 MB/s eta 0:00:00



[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FS = 500

# ── Pick a 10-second window (e.g., seconds 10–20 to skip start artifacts)
start_sec, end_sec = 10, 20
start_idx = start_sec * FS   # 5000
end_idx   = end_sec   * FS   # 10000

# ── Load two subjects: one HC and one patient
# Replace with your actual file loading (mne or pandas)
hc_data  = load_subject(r"D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\Healthy subjects\Healthy Subjects - EEG_csv_format\PNT5_trial.csv")[:, start_idx:end_idx]   # shape (18, 5000)
pd_data  = load_subject(r"D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\AD subjects\AD subjects - EEG_csv_format\sub5_trial.csv")[:, start_idx:end_idx]   # shape (18, 5000)
time_win = np.linspace(start_sec, end_sec, end_idx - start_idx)

CHANNELS = ['Fp1','Fp2','F3','F4','C3','C4','P3','P4',
            'O1','O2','F7','F8','T3','T4','T5','T6','Fz','Cz']

# ── Build figure (18 rows × 2 columns) ─────────────────────────────
fig = make_subplots(rows=18, cols=2, shared_xaxes=True,
                    vertical_spacing=0.003)

for i, ch in enumerate(CHANNELS):
    row = i + 1
    # HC — column 1 (blue)
    fig.add_trace(go.Scatter(x=time_win, y=hc_data[i],
                             mode='lines', line=dict(width=0.8, color='#2196F3'),
                             name='HC', showlegend=(i==0)), row=row, col=1)
    # Patient — column 2 (red)
    fig.add_trace(go.Scatter(x=time_win, y=pd_data[i],
                             mode='lines', line=dict(width=0.8, color='#E53935'),
                             name='Patient', showlegend=(i==0)), row=row, col=2)
    fig.update_yaxes(title_text=ch, title_font=dict(size=9), row=row, col=1)
    fig.update_yaxes(title_text=ch, title_font=dict(size=9), row=row, col=2)

fig.update_xaxes(title_text='Time (s)', row=18, col=1)
fig.update_xaxes(title_text='Time (s)', row=18, col=2)

fig.update_layout(title='EEG Raw Signal: HC vs Patient (10-sec window)',
                  height=1400, width=1100)
fig.show()
fig.write_image("eeg_raw_overlay.png")

NameError: name 'load_subject' is not defined

In [ ]:
#This code is to generate EEG visual comparing Healthy vs Alzheimer subject
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

FS = 500
start_sec, end_sec = 10, 20

# ── Load HC subject ───────────────────────────────────────────────
df_hc = pd.read_csv(r"D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\Healthy subjects\Healthy Subjects - EEG_csv_format\PNT5_trial.csv")   # 210000 rows × 18 columns
hc_window = df_hc.values[start_sec*FS : end_sec*FS, :].T  # shape: (18, 5000)

# ── Load Patient subject ──────────────────────────────────────────
df_pd = pd.read_csv(r"D:\ASDS\Capstone Project\AD Experiment Data in CSV 4\AD Experiment Data in CSV\AD subjects\AD subjects - EEG_csv_format\sub5_trial.csv")
pd_window = df_pd.values[start_sec*FS : end_sec*FS, :].T  # shape: (18, 5000)

CHANNELS = df_hc.columns.tolist()   # reads channel names directly from CSV headers
time_win = np.linspace(start_sec, end_sec, end_sec*FS - start_sec*FS)

fig = make_subplots(rows=len(CHANNELS), cols=2, shared_xaxes=True,
                    vertical_spacing=0.003)

for i, ch in enumerate(CHANNELS):
    row = i + 1
    fig.add_trace(go.Scatter(x=time_win, y=hc_window[i],
                             mode='lines', line=dict(width=0.8, color='#2196F3'),
                             name='HC', showlegend=(i==0)), row=row, col=1)
    fig.add_trace(go.Scatter(x=time_win, y=pd_window[i],
                             mode='lines', line=dict(width=0.8, color='#E53935'),
                             name='Patient', showlegend=(i==0)), row=row, col=2)
    fig.update_yaxes(title_text=ch, title_font=dict(size=9), row=row, col=1)
    fig.update_yaxes(title_text=ch, title_font=dict(size=9), row=row, col=2)

fig.update_xaxes(title_text='Time (s)', row=18, col=1)
fig.update_xaxes(title_text='Time (s)', row=18, col=2)
fig.update_layout(title='EEG Raw Signal: HC vs Patient', height=1400, width=1100)
#fig.show()
fig.write_image("eeg_raw_overlay.png")